# Min semivariance | Research

Objetivo: Encontrar la mejor convinación de activos tales que minimicen la semivarianza de un portafolio con 5 activos utilizando el metodo de minima semivarianza.

Los activos son exclusivamente activos de capital de tal manera que no pueden otro tipo de instrumentos diferentes a:

- Acciones
- ETF de acciones

In [1]:
import pandas as pd
import numpy as np
from itertools import permutations, combinations

from scipy.optimize import minimize

import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

import warnings

warnings.filterwarnings(
    "ignore",
    message=".*Timestamp.utcnow is deprecated.*",
    module="yfinance.*"
)

In [2]:
from src.asset_allocation import PortfolioOptimizationPostModern
from src.asset_allocation import PortfolioPostModernMetrics, MinimumSemivarianceConfig
from src.asset_allocation import PortfolioElementaryMetrics
from src.asset_allocation import PortfolioElementaryAnalysis

from src.security_selection import CorrelationPortfolioSelector

In [3]:
pd.options.display.float_format = '{:.8f}'.format

## Statements de la investigación

Maximo 5 activos de capial que pueden ser acciones o ETF de acciones. Seleccionar un Benchmark adecuado para el portafolio seleccionado consistente con los activos elegicos.

Cuando los activos sean de diferentes paises estos deberan de ser convertidos a la moneda MXN con tal de mantener coherencia en el valor economico de los mismos.

El periodo a evaluar consiste del 2010-01-01 añ 2026-01-01.

### Funtions utils

In [4]:
def buscar_mejor_subportafolio_min_semivarianza_fast(
    portfolio_base,
    r,
    config=None,
    usar_permutaciones=False,
    guardar_evaluaciones=True,
):
    """
    Evaluar todos los subconjuntos de activos de tamaño ``r`` y encontrar
    aquel cuya semivarianza mínima optimizada sea la menor.

    Esta función está pensada como una versión más eficiente que iterar
    creando una instancia nueva de ``PortfolioOptimizationPostModern`` para
    cada combinación. La mejora principal viene de:

    1. Precalcular una sola vez la matriz completa de semivarianza.
    2. Precalcular una sola vez el vector de rendimientos anuales.
    3. Trabajar con submatrices de ``numpy`` en lugar de recortar
       ``DataFrame`` y reconstruir objetos en cada iteración.
    4. Resolver directamente el problema cuadrático de minimización para cada
       subconjunto.

    Parámetros
    ----------
    portfolio_base : PortfolioOptimizationPostModern
        Instancia base que contiene el universo completo de activos y desde la
        cual se obtienen los datos, rendimientos y matriz de semivarianza.
    r : int
        Número de activos por subconjunto a evaluar.
    config : MinimumSemivarianceConfig | None, default None
        Configuración para la optimización. Si es ``None``, se utiliza la
        configuración por defecto de ``MinimumSemivarianceConfig()``.
    usar_permutaciones : bool, default False
        Si es ``True``, evalúa permutaciones de longitud ``r``.
        Si es ``False``, evalúa combinaciones.
        En problemas de portafolios normalmente conviene ``False``, ya que el
        orden de los tickers no cambia el subconjunto económico.
    guardar_evaluaciones : bool, default True
        Si es ``True``, construye y devuelve un ``DataFrame`` con el detalle de
        todas las evaluaciones. Si solo interesa el mejor subconjunto, usar
        ``False`` reduce uso de memoria y mejora el rendimiento.

    Regresa
    -------
    dict
        Diccionario con las llaves:

        - ``"best_tickers"``:
          Lista con los tickers del mejor subconjunto encontrado.
        - ``"best_result"``:
          Diccionario con el detalle de la mejor solución:
          pesos, semivarianza, downside risk, rendimiento esperado, etc.
        - ``"evaluations"``:
          ``DataFrame`` con todas las evaluaciones, solo si
          ``guardar_evaluaciones=True``.

    Notas
    -----
    - Esta función sigue siendo exacta: evalúa todos los subconjuntos.
    - Su costo total continúa siendo combinatorio en el número de activos.
      Si el universo es grande, puede tardar bastante aunque cada iteración
      sea más barata.
    - Si ``config.minimum_return`` está activo, se agrega la restricción
      ``w @ mu >= minimum_return`` al problema de optimización.
    - Si ``config.bounds`` se proporciona, su longitud debe ser exactamente
      ``r``.

    Ejemplo
    -------
    >>> resultado = buscar_mejor_subportafolio_min_semivarianza_fast(
    ...     portfolio_base=portfolio_base,
    ...     r=5,
    ...     config=MinimumSemivarianceConfig(threshold=0.0),
    ...     usar_permutaciones=False,
    ...     guardar_evaluaciones=True,
    ... )
    >>> print(resultado["best_tickers"])
    >>> print(resultado["best_result"]["semivariance"])
    >>> print(resultado["best_result"]["weights"])
    """
    if config is None:
        config = MinimumSemivarianceConfig()

    tickers = np.asarray(portfolio_base.tickers, dtype=object)
    n = len(tickers)

    if not 1 <= r <= n:
        raise ValueError("r debe estar entre 1 y el número de tickers.")

    if config.bounds is not None and len(config.bounds) != r:
        raise ValueError("Si config.bounds no es None, su longitud debe ser r.")

    # En portafolios casi siempre basta con combinaciones, porque
    # [A, B, C] y [C, B, A] representan el mismo subconjunto.
    generador = (
        permutations(range(n), r)
        if usar_permutaciones
        else combinations(range(n), r)
    )

    threshold = float(config.threshold)

    # Precomputamos una sola vez la matriz de semivarianza del universo completo.
    # Luego, para cada subconjunto, solo extraemos la submatriz correspondiente.
    semiv_full = (
        portfolio_base
        .semivariance_matrix(threshold)
        .loc[list(tickers), list(tickers)]
        .to_numpy(dtype=float)
    )

    # Precomputamos una sola vez el vector de rendimientos anuales del universo.
    mu_full = (
        portfolio_base
        .annual_return(list(tickers))
        .loc[list(tickers)]
        .to_numpy(dtype=float)
    )

    # Si el usuario no define cotas, se usan las mismas reglas del optimizador:
    # sin short-selling -> [0, 1]
    # con short-selling -> [-1, 1]
    if config.bounds is not None:
        bounds = list(config.bounds)
    else:
        bounds = [(-1.0, 1.0)] * r if config.allow_short else [(0.0, 1.0)] * r

    # Punto inicial factible para el optimizador.
    # Si no se especifica, usamos pesos equiponderados.
    if config.initial_weights is not None:
        x0 = np.asarray(config.initial_weights, dtype=float)
        if x0.shape != (r,):
            raise ValueError("config.initial_weights debe tener longitud r.")
    else:
        x0 = np.repeat(1.0 / r, r)

    solver_options = dict(config.solver_options)

    rows = [] if guardar_evaluaciones else None
    best = None
    best_semivariance = np.inf

    for idx in generador:
        # Convertimos el subconjunto de índices a arreglo NumPy para extraer
        # submatrices y subvectores de forma eficiente.
        idx = np.fromiter(idx, dtype=int, count=r)

        # Submatriz de semivarianza del subconjunto.
        S = semiv_full[np.ix_(idx, idx)]

        # Vector de rendimientos esperados del subconjunto.
        mu = mu_full[idx]

        # Objetivo cuadrático:
        # minimizar w' S w
        def objective(w, S=S):
            return float(w @ S @ w)

        # Gradiente analítico del objetivo.
        # Entregárselo al solver suele acelerar la convergencia.
        def jacobian(w, S=S):
            return 2.0 * (S @ w)

        # Restricción de presupuesto: la suma de pesos debe ser 1.
        constraints = [
            {
                "type": "eq",
                "fun": lambda w: np.sum(w) - 1.0,
                "jac": lambda w: np.ones_like(w),
            }
        ]

        # Restricción opcional de rendimiento mínimo.
        if config.minimum_return is not None:
            target = float(config.minimum_return)
            constraints.append(
                {
                    "type": "ineq",
                    "fun": lambda w, mu=mu, target=target: float(w @ mu - target),
                    "jac": lambda w, mu=mu: mu,
                }
            )

        solution = minimize(
            fun=objective,
            x0=x0,
            jac=jacobian,
            method=config.solver_method,
            bounds=bounds,
            constraints=constraints,
            options=solver_options,
        )

        success = bool(solution.success)
        weights = np.asarray(solution.x, dtype=float)

        if success:
            semivariance = float(solution.fun)
            expected_return = float(weights @ mu)
            downside_risk = float(np.sqrt(max(semivariance, 0.0)))
        else:
            semivariance = np.inf
            expected_return = np.nan
            downside_risk = np.nan

        subset_tickers = tickers[idx].tolist()

        if guardar_evaluaciones:
            rows.append(
                {
                    "tickers": subset_tickers,
                    "success": success,
                    "semivariance": semivariance,
                    "downside_risk": downside_risk,
                    "expected_return": expected_return,
                    "iterations": int(getattr(solution, "nit", 0)),
                    "message": str(solution.message),
                    "weights": dict(zip(subset_tickers, weights)) if success else None,
                }
            )

        # Actualizamos el mejor resultado encontrado.
        if success and semivariance < best_semivariance:
            best_semivariance = semivariance
            best = {
                "tickers": subset_tickers,
                "weights": pd.Series(weights, index=subset_tickers, name="weight"),
                "semivariance": semivariance,
                "downside_risk": downside_risk,
                "expected_return": expected_return,
                "iterations": int(getattr(solution, "nit", 0)),
                "status": int(solution.status),
                "message": str(solution.message),
            }

    if best is None:
        raise RuntimeError("Ninguna combinación produjo una solución válida.")

    result = {
        "best_tickers": best["tickers"],
        "best_result": best,
    }

    if guardar_evaluaciones:
        result["evaluations"] = (
            pd.DataFrame(rows)
            .sort_values(["success", "semivariance"], ascending=[False, True])
            .reset_index(drop=True)
        )

    return result



## Activos MSCI Minimun Volatility to bouild a portfolio

### Estrategia

Se seleccionarion 50 activos los cuales pertencen a la categoría de MSCI Minimun Volatility donde estos instrumentos estan especificamente diseñados para poder minimar su exposión a la volatilidad relativa de su mercado padre de tal manera que con esta idea se pretende minimizar la escala de la semivarianza downside risk dado que la volatilidad de estos instrumentos se encuentra acotada en rangos mas pequeños creemos que esto nos podría implicar un downside risk global del portafolio menor respecto de otros portafolios. 

A traves de 50 de estos intrumenos, seleccionaremos los 15 que conjuntamente obtengan la menor correlación entre si y los 15 que tenga el menos downside risk y los 15 con mejor relación omega. Una vez seleccionados estos 15 activos para las 3 clasificaciones se permutaran en portafolios con 5 activos de tal manera que se escoga el que menos downside risk total resulte del calculo.

### Seleccion de activos

comenzamos primero generando la metricas para cada activos de tal manera que podamos identificar cuales de estos son los que seran clasificados para la construcción de los portafolios.

In [5]:
start_date = "2010-01-01"
end_date = "2026-01-01"

In [ ]:
# --------------------------------------------
# 1) Universo EE.UU.: mínimo riesgo / mínima varianza (máximo 50)
# 
# --------------------------------------------
minvol_50 = [
    # ETFs defensivos / quality / dividend / low-beta proxies
    "XLP", "XLU", "XLV", "VDC", "VPU", "VHT", "VIG", "VYM", "DVY", "SDY",
    "FDL", "IYK", "IDU", "IYH", "IVE", "IWD", "DIA", "RSP", "VTV",

    # Acciones defensivas / quality / estabilidad de flujo
    "BRK-B", "PG", "KO", "PEP", "JNJ", "WMT", "MCD", "CL", "COST", "GIS",
    "HSY", "MKC", "CLX", "HRL", "SYY", "BDX", "ABT", "MDT", "TMO", "WM",
    "SO", "DUK", "NEE", "ED", "AEP", "XEL", "WEC", "AJG", "MMC", "PGR",
    "ADP"
    
    #Acciones energieticas
    "DOG", "EFZ", "MYY", "SH"

]


In [11]:
# "DOG", "EFZ", "MYY", "SH"
results = ['VDC', 'JNJ', 'MCD', 'ED', 'WMT', "SOXL", "SOXS"]

#'VDC': 

In [ ]:
#
# 2) Activos con la menor correlación de méxico
# 
#

mincorr_assets_mx = ['VALUEGFO.MX', 'KUOB.MX', 'LAMOSA.MX']

In [37]:
results_minvol_50 = [ 'VDC', 'JNJ', 'MCD', 'ED', 'WMT', 'VALUEGFO.MX', 'KUOB.MX', 'LAMOSA.MX']

hedge_structural = [
    "PSQ",     # Short QQQ
    "RWM",     # Short Russell 2000
    "SBB",     # Short SmallCap 600

    # U.S. sectors / industries
    "REK",     # Short Real Estate
    "SBM",     # Short Basic Materials
    "SEF",     # Short Financials

    # International / regional
    "EUM",     # Short MSCI Emerging Markets
    "YXI",     # Short FTSE China 50

    # Alternatives / long-short / arbitrage
    "CSM",     # 130/30 long-short equity
    "MNA",     # Merger arbitrage
    "QAI",     # Hedge multi-strategy tracker

    # Canada listed
    "CNDI.TO", # BetaPro S&P/TSX 60 Daily Inverse ETF
]

hedge_tactical = [
    # Broad market U.S.
    "SDS",     # UltraShort S&P 500 (-2x)
    "SPXU",    # UltraPro Short S&P 500 (-3x)
    "DXD",     # UltraShort Dow 30 (-2x)
    "QID",     # UltraShort QQQ (-2x)
    "TWM",     # UltraShort Russell 2000 (-2x)
    "MZZ",     # UltraShort MidCap 400 (-2x)
    "SDD",     # UltraShort SmallCap 600 (-2x)

    # U.S. sectors / industries
    "SRS",     # UltraShort Real Estate (-2x)
    "SMN",     # UltraShort Materials (-2x)
    "SKF",     # UltraShort Financials (-2x)
    "RXD",     # UltraShort Health Care (-2x)
    "SIJ",     # UltraShort Industrials (-2x)
    "DUG",     # UltraShort Energy / Oil & Gas (-2x)
    "SCC",     # UltraShort Consumer Services (-2x)
    "SZK",     # UltraShort Consumer Goods (-2x)
    "REW",     # UltraShort Technology (-2x)
    "SSG",     # UltraShort Semiconductors (-2x)
    "SDP",     # UltraShort Utilities (-2x)
    "BIS",     # UltraShort Nasdaq Biotechnology (-2x)

    # International / regional
    "EFU",     # UltraShort MSCI EAFE (-2x)
    "EEV",     # UltraShort MSCI Emerging Markets (-2x)
    "EWV",     # UltraShort MSCI Japan (-2x)
    "EPV",     # UltraShort FTSE Europe (-2x)
    "FXP",     # UltraShort FTSE China 50 (-2x)
    "BZQ",     # UltraShort MSCI Brazil (-2x)

    # Direxion tactical bears
    "SPXS",    # S&P 500 Bear 3X
    "TZA",     # Small Cap Bear 3X
    "FAZ",     # Financial Bear 3X
    "TECS",    # Technology Bear 3X
    "EDZ",     # Emerging Markets Bear 3X
    "ERY",     # Energy Bear 2X
    "DRV",     # Real Estate Bear 3X
]


results_minvol_50 = ['VDC', 'JNJ', 'MCD', 'ED', 'WMT',
    "PSQ",     # Short QQQ
    "RWM",     # Short Russell 2000
    "SBB",     # Short SmallCap 600

    # U.S. sectors / industries
    "REK",     # Short Real Estate
    "SBM",     # Short Basic Materials
    "SEF",     # Short Financials

    # International / regional
    "EUM",     # Short MSCI Emerging Markets
    "YXI",     # Short FTSE China 50

    # Alternatives / long-short / arbitrage
    "CSM",     # 130/30 long-short equity
    "MNA",     # Merger arbitrage
    "QAI",     # Hedge multi-strategy tracker

    # Canada listed, # BetaPro S&P/TSX 60 Daily Inverse ETF
    
    # U.S. sectors / industries
    "SRS",     # UltraShort Real Estate (-2x)
    "SMN",     # UltraShort Materials (-2x)
    "SKF",     # UltraShort Financials (-2x)
    "RXD",     # UltraShort Health Care (-2x)
    "SIJ",     # UltraShort Industrials (-2x)
    "DUG",     # UltraShort Energy / Oil & Gas (-2x)
    "SCC",     # UltraShort Consumer Services (-2x)
    "SZK",     # UltraShort Consumer Goods (-2x)
    "REW",     # UltraShort Technology (-2x)
    "SSG",     # UltraShort Semiconductors (-2x)
    "SDP",     # UltraShort Utilities (-2x)
    "BIS",     # UltraShort Nasdaq Biotechnology (-2x)
    
    "QLD", 
    "QID",
    "SOXL", 
    "SOXS"
    ]

31

In [ ]:
n_minvol_50 = len(minvol_50)

weight_minvol_50_0 = np.ones(n_minvol_50) / n_minvol_50

minvol_50_post_metrics = PortfolioPostModernMetrics(tickers=minvol_50, start=start_date, end=end_date, weight=weight_minvol_50_0)

In [10]:
omega_minvol_50 = pd.DataFrame(
    minvol_50_post_metrics.asset_omega_ratio(),
    columns=["omega"],
).rename_axis("ticker")

downside_risk_minvol_50 = pd.DataFrame(
    minvol_50_post_metrics._downside_risk_series(), 
    columns=["downside_risk"]
    ).rename_axis("ticker")

upside_risk_minvol_50 = pd.DataFrame(
    minvol_50_post_metrics._upside_risk_series(),
    columns=["upside_risk"]
).rename_axis("ticker")


metrics_minvol_50 = pd.concat(
    [
        omega_minvol_50, 
        downside_risk_minvol_50, 
        upside_risk_minvol_50
    ], 
    axis = 1
    )

metrics_minvol_50.sort_values(by = "downside_risk", ascending=True).head(5)

,omega,downside_risk,upside_risk
ticker,,,
VDC,0.977281,0.085093,0.083160
XLP,0.966973,0.085975,0.083136
IYK,0.917560,0.094890,0.087067
VIG,0.962834,0.098240,0.094589
FDL,0.956365,0.098839,0.094526


In [11]:
mim_vol_50_corr = CorrelationPortfolioSelector(start_date=start_date, end_date=end_date)

group_min_vol = {
    "min_vol_50" : minvol_50
}

mim_vol_50_corr_results = mim_vol_50_corr.rank_within_groups(grouped_tickers=group_min_vol, top_k=50)
mim_vol_50_corr_results.set_index(keys="ticker", inplace=True)
mim_vol_50_corr_results.drop(columns=["group", "selected"], inplace=True)

In [12]:
mim_vol_50_corr_results.head(5)

,corr_score,rank_in_group
ticker,,
CLX,0.350900,1
WMT,0.385079,2
HRL,0.397947,3
GIS,0.416315,4
SYY,0.424351,5


Una ves obtenidas las metricas necesarias para todos los activos solo es cuestion de ordenarlos con los criterios anteriore y seleccionarlos.

In [ ]:
n_assets = 10

minvol_50_min_corr_assets = mim_vol_50_corr_results.index[0:n_assets]
minvol_50_min_downside_risk_assets = metrics_minvol_50.sort_values(by = "downside_risk", ascending=True).index[0:n_assets]
minvol_50_max_omega_assets = metrics_minvol_50.sort_values(by = "omega", ascending=False).index[0:n_assets]

#### Min correlacion

Con esto ya podemos generar las permutaciones para poder evaluar las medidas

In [14]:
initial_weight_min_corr = np.ones(len(minvol_50_min_corr_assets)) / len(minvol_50_min_corr_assets)

portafolio_base_min_corr = PortfolioOptimizationPostModern(tickers=list(minvol_50_min_corr_assets), start=start_date, end=end_date, weight=initial_weight_min_corr)

config_min_corr = MinimumSemivarianceConfig(threshold=0.0)

result_min_corr = buscar_mejor_subportafolio_min_semivarianza_fast(portfolio_base=portafolio_base_min_corr, r = 5, config=config_min_corr, usar_permutaciones=False)

result_min_corr

{'best_tickers': ['CLX', 'WMT', 'BDX', 'MCD', 'ED'],
 'best_result': {'tickers': ['CLX', 'WMT', 'BDX', 'MCD', 'ED'],
  'weights': CLX    0.144293
  WMT    0.239154
  BDX    0.128381
  MCD    0.284213
  ED     0.203959
  Name: weight, dtype: float64,
  'semivariance': 0.006855758798574948,
  'downside_risk': 0.08279950965177843,
  'expected_return': 0.1238596686638043,
  'iterations': 9,
  'status': 0,
  'message': 'Optimization terminated successfully'},
 'evaluations':                         tickers  success  semivariance  downside_risk  \
 0      [CLX, WMT, BDX, MCD, ED]     True      0.006856       0.082800   
 1      [WMT, GIS, BDX, MCD, ED]     True      0.006882       0.082961   
 2      [CLX, WMT, GIS, MCD, ED]     True      0.006900       0.083066   
 3      [CLX, WMT, HSY, MCD, ED]     True      0.006939       0.083301   
 4      [CLX, WMT, PGR, MCD, ED]     True      0.006945       0.083338   
 ...                         ...      ...           ...            ...   
 2998  [

In [15]:
initial_weight_result_corr = np.ones(len(result_min_corr["best_tickers"])) / len(result_min_corr["best_tickers"])

Portfolio_min_corr_result = PortfolioOptimizationPostModern(tickers=result_min_corr["best_tickers"], start=start_date, end=end_date, weight=initial_weight_result_corr)

result_corr_validation = Portfolio_min_corr_result.optimize_minimum_semivariance()

result_corr_validation.downside_risk

0.08279950965173964

#### Min downside risk

In [38]:
initial_weight_downside_risk = np.ones(len(results_minvol_50)) / len(results_minvol_50)

portafolio_base_downside_risk = PortfolioOptimizationPostModern(tickers=list(results_minvol_50), start=start_date, end=end_date, weight=initial_weight_downside_risk)

config_downside_risk = MinimumSemivarianceConfig(threshold=0.0)

result_downside_risk = buscar_mejor_subportafolio_min_semivarianza_fast(portfolio_base=portafolio_base_downside_risk, r = 5, config=config_downside_risk, usar_permutaciones=False)

result_downside_risk

{'best_tickers': ['PSQ', 'QLD', 'QID', 'SOXL', 'SOXS'],
 'best_result': {'tickers': ['PSQ', 'QLD', 'QID', 'SOXL', 'SOXS'],
  'weights': PSQ    0.30851250
  QLD    0.38694087
  QID    0.26014091
  SOXL   0.02204681
  SOXS   0.02235892
  Name: weight, dtype: float64,
  'semivariance': 1.281147083230257e-05,
  'downside_risk': 0.0035793115025522116,
  'expected_return': -0.018733634162965324,
  'iterations': 18,
  'status': 0,
  'message': 'Optimization terminated successfully'},
 'evaluations':                             tickers  success  semivariance  downside_risk  \
 0       [PSQ, QLD, QID, SOXL, SOXS]     True    0.00001281     0.00357931   
 1         [PSQ, SEF, CSM, QLD, QID]     True    0.00001345     0.00366754   
 2         [PSQ, REK, CSM, QLD, QID]     True    0.00001359     0.00368658   
 3         [PSQ, CSM, SKF, QLD, QID]     True    0.00001361     0.00368863   
 4         [PSQ, EUM, QAI, QLD, QID]     True    0.00001361     0.00368909   
 ...                             ..

In [ ]:
initial_weight_result_downside = np.ones(len(result_downside_risk["best_tickers"])) / len(result_downside_risk["best_tickers"])

Portfolio_min_downside_risk_result = PortfolioOptimizationPostModern(tickers=result_downside_risk["best_tickers"], start=start_date, end=end_date, weight=initial_weight_result_downside)

result_downside_validation = Portfolio_min_downside_risk_result.optimize_minimum_semivariance()

#0.012290924022976274
#0.012648661977122047
#0.003487167148936707
#0.003399643690119102
result_downside_validation.downside_risk

0.003399643690119102

#### Max omega

In [18]:
initial_weight_omega = np.ones(len(minvol_50_max_omega_assets)) / len(minvol_50_max_omega_assets)

portafolio_base_omega = PortfolioOptimizationPostModern(tickers=list(minvol_50_max_omega_assets), start=start_date, end=end_date, weight=initial_weight_omega)

config_omega = MinimumSemivarianceConfig(threshold=0.0)

result_omega = buscar_mejor_subportafolio_min_semivarianza_fast(portfolio_base=portafolio_base_omega, r = 5, config=config_omega, usar_permutaciones=False)

result_omega

{'best_tickers': ['MCD', 'WMT', 'JNJ', 'PG', 'DUK'],
 'best_result': {'tickers': ['MCD', 'WMT', 'JNJ', 'PG', 'DUK'],
  'weights': MCD    0.216306
  WMT    0.207102
  JNJ    0.262364
  PG     0.164855
  DUK    0.149373
  Name: weight, dtype: float64,
  'semivariance': 0.006625512215320775,
  'downside_risk': 0.0813972494333855,
  'expected_return': 0.12651244433016479,
  'iterations': 13,
  'status': 0,
  'message': 'Optimization terminated successfully'},
 'evaluations':                            tickers  success  semivariance  downside_risk  \
 0         [MCD, WMT, JNJ, PG, DUK]     True      0.006626       0.081397   
 1          [MCD, WMT, SO, JNJ, PG]     True      0.006660       0.081609   
 2        [MCD, WMT, HSY, JNJ, DUK]     True      0.006661       0.081614   
 3         [MCD, WMT, HSY, JNJ, PG]     True      0.006671       0.081676   
 4         [MCD, WMT, JNJ, DUK, CL]     True      0.006672       0.081682   
 ...                            ...      ...           ...     

In [19]:
initial_weight_result_omega = np.ones(len(result_omega["best_tickers"])) / len(result_omega["best_tickers"])

Portfolio_max_omega_result = PortfolioOptimizationPostModern(tickers=result_omega["best_tickers"], start=start_date, end=end_date, weight=initial_weight_result_omega)

result_omega_validation = Portfolio_max_omega_result.optimize_minimum_semivariance()

result_omega_validation.downside_risk

0.08139724943351466

Con los resultados obtenidos podemos apreciar que la mejor forma de obtener un portafolio con minima semivarianza es a través de optimizar justamente este parametro en las combinaciones que queremos hacer

### Presentacion de resultados

In [34]:
# Selecciona aqui cual de los resultados quieres revisar como portafolio final.

minvol_50_final_tickets = result_downside_risk["best_tickers"]
# minvol_50_final_tickets = result_min_corr["best_tickers"]
# minvol_50_final_tickets = result_omega["best_tickers"]

#minvol_50_final_tickets = ['NI', 'EVRG', 'PEP', 'VDC', 'EMR']

#=============
#FINAL RESULTS | Min Downside Risk
#=============
#['VDC', 'JNJ', 'MCD', 'ED', 'WMT']
#minvol_50_final_tickets = ['VDC', 'LAMOSA.MX', 'GENTERA.MX', 'EFZ', 'XEF.TO']


minvol_initial_weight_final = np.ones(len(minvol_50_final_tickets)) / len(minvol_50_final_tickets)

PortfolioFinal = PortfolioOptimizationPostModern(
    tickers=minvol_50_final_tickets,
    start=start_date,
    end=end_date,
    weight=minvol_initial_weight_final)


res_final = PortfolioFinal.optimize_minimum_semivariance()

res_final.downside_risk

0.003487167148936707

In [35]:
PortfolioFinal.portfolio_downside_risk()**2/252

4.825529652628318e-08

In [18]:
res_final.weights

array([0.05723529, 0.02699949, 0.02130588, 0.45035845, 0.44410088])

## Activos generales

In [45]:
# --------------------------------------------
# 2) Universo EE.UU.: baja correlación estructural (máximo 100)
# Construido para dispersar por:
# - tamaño / estilo
# - sectores U.S.
# - regiones / países / mercados
# - dividendos / value / growth
# all vía ETFs de equity viejos y líquidos.
# --------------------------------------------
us_lowcorr_100 = [
    # Broad / size / style / dividend U.S.
    "SPY", "DIA", "QQQ", "MDY", "IWM", "IJR", "IJH", "IVE", "IVW", "IWD",
    "IWF", "IWN", "IWO", "IWS", "IWP", "IWB", "IYY", "RSP", "VTI", "VTV",
    "VUG", "VOE", "VOT", "VBR", "VBK", "VYM", "VIG", "DVY", "SDY", "FDL",

    # Sectores U.S. (SPDR / iShares / Vanguard)
    "XLB", "XLE", "XLF", "XLI", "XLK", "XLP", "XLU", "XLV", "XLY", "IYC",
    "IYE", "IYF", "IYH", "IYJ", "IYK", "IYM", "IYR", "IYT", "IYW", "IYZ",
    "VAW", "VCR", "VDC", "VDE", "VFH", "VGT", "VHT", "VIS", "VPU", "VOX",

    # Internacional / países / regiones (todos equity ETFs listados en EE.UU.)
    "EFA", "EEM", "EPP", "EWA", "EWC", "EWG", "EWH", "EWI", "EWJ", "EWK",
    "EWL", "EWM", "EWN", "EWO", "EWP", "EWQ", "EWS", "EWT", "EWU", "EWW",
    "EWY", "EWZ", "EZA", "FXI", "ILF", "FEZ", "ECH", "TUR", "THD", "EPI",
    "IDX", "GXC", "EWD", "EIS", "EPU", "IEV", "RWX", "IFGL", "SCZ", "ACWI"
]

## Activos de alto crecimiento

In [46]:
# --------------------------------------------
# 3) Universo EE.UU.: alto riesgo + retornos extraordinarios (máximo 50)
# Aquí sí busco ganadores agresivos, pero que existan durante toda la ventana.
# --------------------------------------------
us_highreturn_50 = [
    # ETFs agresivos / growth / tech / semis / internet
    "QQQ", "VGT", "XLK", "IYW", "SOXX", "IGV", "FDN", "PNQI", "PSJ", "XSD",

    # Acciones de alto crecimiento de larga data
    "AAPL", "MSFT", "NVDA", "AMZN", "NFLX", "MELI", "INTU", "ISRG", "BKNG", "UNH",
    "ORLY", "AZO", "CTAS", "LULU", "DECK", "MPWR", "AVGO", "ADBE", "CRM", "KLAC",
    "LRCX", "AMAT", "CDNS", "SNPS", "FTNT", "MA", "V", "MSCI", "FICO", "TMO",
    "ROP", "FAST", "GWW", "ROST", "SHW", "ODFL", "VRTX", "ALGN", "MNST", "AMD"
]


## Otros Paises

In [ ]:
mex_minvar_50minvol_50 = [
    "NAFTRACISHRS.MX",
    "AMXB.MX",
    "WALMEX.MX",
    "FEMSAUBD.MX",
    "AC.MX",
    "BIMBOA.MX",
    "KOFUBL.MX",
    "GFNORTEO.MX",
    "GENTERA.MX",
    "BBAJIOO.MX",
    "BOLSAA.MX",
    "GAPB.MX",
    "ASURB.MX",
    "OMAB.MX",
    "MEGACPO.MX",
    "LABB.MX",
    "CHDRAUIB.MX",
    "HERDEZ.MX",
    "KIMBERA.MX",
    "GRUMAB.MX",
    "GCC.MX",
    "Q.MX",
    "LACOMERUBC.MX",
    "FRAGUAB.MX",
    "VESTA.MX",
    "TLEVISACPO.MX",
    "CUERVO.MX",
    "ALSEA.MX",
    "LIVEPOLC-1.MX",
    "ORBIA.MX",
    "ALPEKA.MX",
    "PINFRA.MX",
    "TRAXIONA.MX",
    "RA.MX",
    "GFINBURO.MX",
    "MONEXB.MX",
    "GCARSOA1.MX",
    "MEDICAB.MX",
    "AGUA.MX",
    "GISSA.MX",
    "ALFAA.MX",
    "PEOLES.MX",
    "GMEXICOB.MX",
    "CEMEXCPO.MX",
    "AUTLANB.MX",
    "VITROA.MX",
    "VOLARA.MX",
    "GMXT.MX",
    "ELEKTRA.MX",
    "CYDSASAA.MX",
]